# Verify the public artifact inventory

The top-level manifest records one SHA-256 value per committed computational artifact. This notebook verifies every hash and inspects the public file types.

In [ ]:
import json
from collections import Counter
from pathlib import Path

from experiments.verify_artifacts import verify_manifest

repo_root = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'experiments' / 'results').is_dir()
)
manifest_path = repo_root / 'experiments/results/manifest.json'
errors = verify_manifest(manifest_path)
assert not errors, '\n'.join(errors)

In [ ]:
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
paths = sorted(manifest['artifact_sha256'])
suffix_counts = Counter(Path(path).suffix for path in paths)
print('verified artifacts:', len(paths))
print('file types:', dict(sorted(suffix_counts.items())))
print('seed:', manifest['base_seed'])
print('selection:', manifest['selection'])

In [ ]:
allowed = {'.csv', '.json', '.svg'}
assert set(suffix_counts) <= allowed
assert not any(path.startswith(('tex/', 'paper/', 'manuscript/')) for path in paths)
assert manifest['selection'] == 'all'

A passing hash check proves integrity of the committed inventory. Scientific equivalence after changing an implementation still requires the unit tests and validation residuals.